In [ ]:
import os 
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI
from IPython.core import async_helpers
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import time

load_dotenv(override=True)
base_url = os.getenv('OLLAMA_BASE_URL')

def fetch_with_selenium(url, wait_time=3, headless=True):
    """Sayfayi gercek bir Chrome ile render eder ve (baslik, metin) dondurur."""
    options = Options()
    if headless:
        # Bazi siteler headless tarayicilari engeller. Sayfa bos gelirse
        # asagidaki satiri yorum yaparak (headless=False) tekrar deneyin.
        options.add_argument("--headless=new")
        options.add_argument("--disable-gpu")
        options.add_argument("--no-sandbox")
        options.add_argument("--window-size=1920,1080")
        options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(options=options)
    try:
        driver.get(url)
        time.sleep(wait_time)  
        html = driver.page_source
    finally:
        driver.quit()  

    soup = BeautifulSoup(html, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return title, text



football_system_prompt = """
You are an experienced football reporter and commentator. You will be given raw text extracted from a football club's website. Analyze this text and summarize it in Turkish,
markdown format. Ignore menu/navigation text, cookie warnings, and
advertisements. Do not wrap your answer in code blocks; just provide markdown.

Use the following headings (only if relevant information is available):
- **About the team**: Brief general introduction/status
- **Latest news**: Featured news and announcements (point by point)
- **Team status**: Information such as form, recent match results, league/standings
"""
ollama=OpenAI(base_url=base_url,api_key="ollama")

def summarize_team(url, headless=True):
    title, text = fetch_with_selenium(url, headless=headless)
    user_prompt = (
        f"Here is the content of the football club page titled '{title}':"
         "Briefly summarize the team, the latest news, and the team's current situation.\n\n"
        + text[:6000]  
    )
    response = ollama.chat.completions.create(
        model="llama3.2:ib",
        messages=[
            {"role": "system", "content": football_system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    return response.choices[0].message.content


def display_team_summary(url, headless=True):
    display(Markdown(summarize_team(url, headless=headless)))


team_url = "https://www.transfermarkt.com.tr/fenerbahce-istanbul/startseite/verein/36"
display_team_summary(team_url)

APITimeoutError: Request timed out.